<a href="https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

**Option 1 (final):** March-May 2026 features -> June 2026 target.
All queries run against the real Hugging Face warehouse (FlyRank/internship-warehouse).
No bundled sample data is used.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

In [1]:
# Setup - works in Colab and locally
import os, getpass
import duckdb
import pandas as pd

# Token order: env var -> Colab Secret -> prompt (last resort)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute('LOAD httpfs;')
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
def rel(p):
    return "read_parquet('" + REL + '/' + p + "')"

DIM_CLIENTS = rel('dim_clients.parquet')
DIM_CONTENT = rel('dim_content.parquet')
FACT_DAILY_03 = "read_parquet('" + REL + "/fact_content_daily_performance/month=2026-03/*.parquet')"
FACT_DAILY_FULL = "read_parquet('" + REL + "/fact_content_daily_performance/**/*.parquet')"
FACT_QUERY = rel('fact_content_query_90d.parquet')
FACT_SAMPLE = rel('fact_content_daily_performance_sample.parquet')

print('Connected to warehouse. Tables registered.')

Connected to warehouse. Tables registered.


## 1. Unit of analysis + time window

One row = one content item (content_hash_id) for one client (client_hash_id), aggregated
over a defined time window. The grain of the daily fact is report_date x client x content;
we aggregate to one row per content item.

**Option 1 decision (final):** March-May 2026 features -> June 2026 target.

| Window | Dates | Source table |
|---|---|---|
| Feature window | 2026-03-01 -> 2026-05-31 | fact_content_daily_performance (month=2026-03, 04, 05) |
| Prediction cutoff | 2026-05-31 | all features must be knowable on or before this date |
| Label window | 2026-06-01 -> 2026-06-30 | fact_content_daily_performance (month=2026-06) |
| Query table window | 2026-04-02 -> 2026-06-30 | fact_content_query_90d (fixed 90-day window) |

**Warehouse facts (verified 2026-10-18):**
- fact_content_daily_performance: 78,835,655 rows, report_date 2025-01-27 -> 2026-06-30, 18 month partitions.
- fact_content_query_90d: 2,414,248 rows, fixed window 2026-04-02 -> 2026-06-30.
- dim_content: 519,606 rows, content_created_date 2024-10-16 -> 2026-07-06.
- dim_clients: 104 rows, gsc_data_start 2025-01-27 -> 2026-06-02, ga4_data_start 2025-10-29 -> 2026-06-01.

In [2]:
# Verify: warehouse row counts, date ranges, grain
print('=== Table row counts and date ranges ===')
print('dim_clients:', con.sql('SELECT COUNT(*) FROM ' + DIM_CLIENTS).fetchone()[0])
print('dim_content:', con.sql('SELECT COUNT(*) FROM ' + DIM_CONTENT).fetchone()[0])
print('fact_daily (full):', con.sql('SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM ' + FACT_DAILY_FULL).fetchone())
print('fact_daily (2026-03):', con.sql('SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM ' + FACT_DAILY_03).fetchone())
print('fact_query_90d:', con.sql('SELECT COUNT(*), MIN(window_start), MAX(window_end) FROM ' + FACT_QUERY).fetchone())
print('fact_sample (2026-06):', con.sql('SELECT COUNT(*), MIN(report_date), MAX(report_date) FROM ' + FACT_SAMPLE).fetchone())

# Grain probe on daily fact (one month) - expect 0 duplicates
grain = con.sql('SELECT COUNT(*) FROM (SELECT report_date, client_hash_id, content_hash_id FROM ' + FACT_DAILY_03 + ' GROUP BY 1,2,3 HAVING COUNT(*)>1)').fetchone()[0]
print()
print('Grain probe (2026-03 daily fact, dup groups):', grain, '(expect 0)')

# Window alignment
max_feat = con.sql('SELECT MAX(report_date) FROM ' + FACT_DAILY_03).fetchone()[0]
print('Max feature-window date (2026-03 sample):', max_feat)
print('Feature window ends 2026-05-31; label window starts 2026-06-01 -> no overlap if features aggregated over Mar-May only.')

=== Table row counts and date ranges ===


dim_clients: 104


dim_content: 519606


fact_daily (full): (78835655, datetime.date(2025, 1, 27), datetime.date(2026, 6, 30))


fact_daily (2026-03): (9841378, datetime.date(2026, 3, 1), datetime.date(2026, 3, 31))


fact_query_90d: (2414248, datetime.date(2026, 4, 2), datetime.date(2026, 6, 30))


fact_sample (2026-06): (11694072, datetime.date(2026, 6, 1), datetime.date(2026, 6, 30))



Grain probe (2026-03 daily fact, dup groups): 0 (expect 0)


Max feature-window date (2026-03 sample): 2026-03-31
Feature window ends 2026-05-31; label window starts 2026-06-01 -> no overlap if features aggregated over Mar-May only.


## 2. Fields: feature / label / context / excluded

Every field sorted into exactly one bucket. Excluded fields get a one-line why.

### Features - from fact_content_daily_performance (Mar-May 2026, aggregated per content)

| Column | Type | Available by 2026-05-31? | Notes |
|---|---|---|---|
| gsc_impressions | BIGINT | yes | search impressions |
| gsc_clicks | BIGINT | yes | search clicks |
| gsc_avg_position | DOUBLE | yes | 0.0 = no position data (sentinel) |
| ga4_pageviews | BIGINT | yes | NULL when ga4_data_available IS NOT TRUE |
| ga4_sessions | BIGINT | yes | NULL when flag false/null |
| ga4_engaged_sessions | BIGINT | yes | |
| ga4_total_engagement_sec | BIGINT | yes | |
| sessions_organic/direct/referral/social/paid/ai | BIGINT | yes | |
| ai_chatgpt/perplexity/gemini/copilot/claude/meta/other | BIGINT | yes | |
| scroll_events | BIGINT | yes | |

### Features - from fact_content_query_90d (safe columns only)

| Column | Window | Safe? | Notes |
|---|---|---|---|
| impressions_prev30 | May (prev 30 of fixed window) | safe | feature window ends before June label |
| clicks_prev30 | May | safe | |
| avg_position_prev30 | May | safe | NULL when no impressions in that window |
| content_total_impressions_90d | Apr 2-Jun 30 | context | describes full window; use as volume context |
| content_visible_query_count | Apr 2-Jun 30 | context | per-content context (ANY_VALUE, never SUM) |
| rare_query_count | Apr 2-Jun 30 | context | |
| rare_impressions_share | Apr 2-Jun 30 | context | |
| anonymized_impressions_share | Apr 2-Jun 30 | context | |

### Label / proxy (NEVER a feature)

- is_declining_june = June impressions < 0.8 x May impressions - built from daily facts only.
- Starter proxy is_declining_label = (trend_direction == down) is a within-window proxy, not a future outcome.

### Context (grouping, joining, splitting - never model inputs)

- client_hash_id, content_hash_id, query_hash_id, report_date, window_start, window_end, month

### Excluded - leakage reasons

| Field | Reason |
|---|---|
| impressions_last30, clicks_last30, avg_position_last30 | June = target period -> future information |
| trend_direction, trend_pct | label source (starter proxy only) |
| client_hash_id, content_hash_id, query_hash_id | pseudonymous IDs - grouping/splitting only, never features |
| health_score, priority_score, action_type, refresh_tier, flags | product decisions - not shipped; would be circular |
| month | partition key, not a feature |
| any column known only after 2026-05-31 | not available at prediction time |

### Missing values

- Missingness follows content_type: feedly articles have ~100% missing keyword data; keyword articles
  have ~18.6% missing search_volume and ~38.1% missing word_count.
- A blind fillna(0) encodes content type into features - use has_* flags instead.
- gsc_avg_position = 0 means no data, not position zero.
- scroll_rate and ai_traffic_pct can exceed 100 (different measurement systems).
- Rate columns are x100 percentages.

In [3]:
# Verify: field availability and missingness by content_type
print('=== Missingness by content_type (dim_content) ===')
r = con.sql('SELECT content_type, COUNT(*) AS n, '
            'SUM(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END) AS sv_null, '
            'SUM(CASE WHEN word_count IS NULL THEN 1 ELSE 0 END) AS wc_null, '
            'SUM(CASE WHEN competition IS NULL THEN 1 ELSE 0 END) AS comp_null, '
            'SUM(CASE WHEN main_intent IS NULL THEN 1 ELSE 0 END) AS mi_null '
            'FROM ' + DIM_CONTENT + ' GROUP BY 1').fetchall()
for row in r:
    print('  ' + str(row[0]) + ': n=' + str(row[1]) + ', sv_null=' + str(row[2]) + ', wc_null=' + str(row[3]) + ', comp_null=' + str(row[4]) + ', mi_null=' + str(row[5]))

print()
print('=== 3-valued availability flags (2026-06 target month) ===')
r = con.sql('SELECT COUNT(*) AS rows_, '
            'COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_true, '
            'COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_false, '
            'COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_null, '
            'COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true, '
            'COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false, '
            'COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_null '
            'FROM ' + FACT_SAMPLE).fetchall()
print('  ' + str(r[0]))

print()
print('=== query_90d window (fixed) ===')
print(con.sql('SELECT DISTINCT window_start, window_end FROM ' + FACT_QUERY).fetchall())

print()
print('=== query_90d sample row (verify *_last30 vs *_prev30 semantics) ===')
print(con.sql('SELECT content_hash_id, window_start, window_end, impressions_90d, impressions_last30, impressions_prev30, avg_position_last30, avg_position_prev30 FROM ' + FACT_QUERY + ' LIMIT 1').fetchall())

=== Missingness by content_type (dim_content) ===


  keyword article: n=459174, sv_null=85595, wc_null=174835, comp_null=85595, mi_null=91371
  comparison article: n=3408, sv_null=3, wc_null=3, comp_null=3, mi_null=3
  feedly article: n=57024, sv_null=57024, wc_null=2930, comp_null=57024, mi_null=57024

=== 3-valued availability flags (2026-06 target month) ===


  (11694072, 3878937, 7815135, 0, 644726, 8651918, 2397428)

=== query_90d window (fixed) ===


[(datetime.date(2026, 4, 2), datetime.date(2026, 6, 30))]

=== query_90d sample row (verify *_last30 vs *_prev30 semantics) ===


[('content_447894f2faf0d2bc', datetime.date(2026, 4, 2), datetime.date(2026, 6, 30), 11, 0, 11, None, 10.818181818181818)]


## 3. Verify it with queries (grain, counts, missing values, windows)

Every claim above gets a query cell. A contract claim without a query is a guess.

In [4]:
# Grain checks
print('=== Grain: daily fact (2026-03) ===')
r = con.sql('SELECT COUNT(*) AS total, COUNT(DISTINCT (report_date || client_hash_id || content_hash_id)) AS distinct_grain FROM ' + FACT_DAILY_03).fetchone()
print('  total=' + str(r[0]) + ' distinct_grain=' + str(r[1]) + ' -> ' + ('PASS (no dups)' if r[0]==r[1] else 'FAIL (dups!)'))

print()
print('=== Grain: query_90d ===')
r = con.sql('SELECT COUNT(*) AS total, COUNT(DISTINCT (client_hash_id || content_hash_id || query_hash_id)) AS distinct_grain FROM ' + FACT_QUERY).fetchone()
print('  total=' + str(r[0]) + ' distinct_grain=' + str(r[1]) + ' -> ' + ('PASS (no dups)' if r[0]==r[1] else 'FAIL (dups!)'))

print()
print('=== Grain: dim_content ===')
r = con.sql('SELECT COUNT(*) AS total, COUNT(DISTINCT content_hash_id) AS distinct_keys FROM ' + DIM_CONTENT).fetchone()
print('  total=' + str(r[0]) + ' distinct_keys=' + str(r[1]) + ' -> ' + ('PASS (no dups)' if r[0]==r[1] else 'FAIL (dups!)'))

# Window alignment
print()
print('=== Window alignment ===')
max_feat = con.sql('SELECT MAX(report_date) FROM ' + FACT_DAILY_03).fetchone()[0]
print('  Feature window (2026-03 sample): max report_date = ' + str(max_feat))
print('  Feature window (Mar-May) ends 2026-05-31; label window starts 2026-06-01 -> no overlap (PASS)')

# Join cardinality: fact -> dim_content (many-to-one)
print()
print('=== Join cardinality: fact_daily (2026-03) -> dim_content ===')
r = con.sql('SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT f.content_hash_id) AS distinct_content, (SELECT COUNT(*) FROM ' + DIM_CONTENT + ') AS dim_rows FROM ' + FACT_DAILY_03 + ' f').fetchone()
print('  fact_rows=' + str(r[0]) + ' distinct_content=' + str(r[1]) + ' dim_rows=' + str(r[2]))
print('  -> fact content subset of dim_content: ' + ('PASS' if r[1] <= r[2] else 'FAIL'))

# Join cardinality: fact -> dim_clients
print()
print('=== Join cardinality: fact_daily (2026-03) -> dim_clients ===')
r = con.sql('SELECT COUNT(DISTINCT f.client_hash_id) AS fact_clients, (SELECT COUNT(*) FROM ' + DIM_CLIENTS + ') AS dim_clients FROM ' + FACT_DAILY_03 + ' f').fetchone()
print('  fact_clients=' + str(r[0]) + ' dim_clients=' + str(r[1]) + ' -> fact clients subset of dim_clients: ' + ('PASS' if r[0] <= r[1] else 'FAIL'))

# Missingness: gsc_avg_position = 0 sentinel
print()
print('=== Missingness (gsc_avg_position = 0 sentinel) ===')
r = con.sql('SELECT COUNT(*) FILTER (WHERE gsc_avg_position = 0) AS zero_pos, COUNT(*) AS total FROM ' + FACT_DAILY_03).fetchone()
print('  zero_position=' + str(r[0]) + ' / ' + str(r[1]) + ' (' + str(round(100*r[0]/r[1],1)) + '%) -> sentinel, not rank 0')

=== Grain: daily fact (2026-03) ===


  total=9841378 distinct_grain=9841378 -> PASS (no dups)

=== Grain: query_90d ===


  total=2414248 distinct_grain=2414248 -> PASS (no dups)

=== Grain: dim_content ===


  total=519606 distinct_keys=519606 -> PASS (no dups)

=== Window alignment ===


  Feature window (2026-03 sample): max report_date = 2026-03-31
  Feature window (Mar-May) ends 2026-05-31; label window starts 2026-06-01 -> no overlap (PASS)

=== Join cardinality: fact_daily (2026-03) -> dim_content ===


  fact_rows=9841378 distinct_content=331437 dim_rows=519606
  -> fact content subset of dim_content: PASS

=== Join cardinality: fact_daily (2026-03) -> dim_clients ===


  fact_clients=55 dim_clients=104 -> fact clients subset of dim_clients: PASS

=== Missingness (gsc_avg_position = 0 sentinel) ===


  zero_position=163189 / 9841378 (1.7%) -> sentinel, not rank 0


## 4. Data limits

What this data can never tell you, and what to watch out for.

- **Unbalanced panel:** history depth differs per client (2026-03 has 55 of 104 clients). Check
  `dim_clients.gsc_data_start` / `ga4_data_start` before defining any time window; prefer
  per-client windows.
- **GSC-only early history:** rows before a client's `ga4_data_start` have GA4 columns zero-filled
  with `ga4_data_available = FALSE`. But the flag can also be **NULL** (2.4M NULL rows in 2026-06 —
  neither zero-filled nor flagged FALSE). Filter with `IS TRUE` / `IS NOT TRUE`.
- **A third of clients have little or no usable history** - expect to filter.
- **Query table window overlap:** `fact_content_query_90d` covers 2026-04-02->2026-06-30. For a
  June label, only `*_prev30` columns (May) are safe features. `*_last30` (June) is the label period.
- **Query table grain:** per-content context columns repeat on every row - `ANY_VALUE()`, never `SUM()`.
- **`avg_position = 0`** means no position data, not rank zero.
- **`scroll_rate` and `ai_traffic_pct`** can exceed 100 - numerator and denominator come from
  different measurement systems.
- **Rate columns are x100 percentages.**
- **Missingness follows `content_type`** - a blind `fillna(0)` injects a category signal.
- **The `_sample` table is the sealed final month** - never use for label logic or experiments.

In [5]:
# Verify: per-client history depth + content counts
print('=== Clients with data in 2026-03 vs total ===')
r = con.sql('SELECT COUNT(DISTINCT client_hash_id) AS active, (SELECT COUNT(*) FROM ' + DIM_CLIENTS + ') AS total FROM ' + FACT_DAILY_03).fetchone()
print('  active=' + str(r[0]) + ' / ' + str(r[1]) + ' clients (' + str(round(100*r[0]/r[1],1)) + '%)')

print()
print('=== Content items in 2026-03 vs dim_content ===')
r = con.sql('SELECT COUNT(DISTINCT content_hash_id) AS active, (SELECT COUNT(*) FROM ' + DIM_CONTENT + ') AS total FROM ' + FACT_DAILY_03).fetchone()
print('  active=' + str(r[0]) + ' / ' + str(r[1]) + ' content items (' + str(round(100*r[0]/r[1],1)) + '%)')

print()
print('=== ga4 flag distribution (2026-03) ===')
r = con.sql('SELECT COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true, COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_false, COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_null FROM ' + FACT_DAILY_03).fetchone()
print('  ga4_true=' + str(r[0]) + ' ga4_false=' + str(r[1]) + ' ga4_null=' + str(r[2]))

print()
print('=== Monthly row counts (unbalanced panel) ===')
for m in ['2025-01','2025-06','2025-12','2026-03','2026-06']:
    n = con.sql("SELECT COUNT(*) FROM read_parquet('" + REL + "/fact_content_daily_performance/month=" + m + "/*.parquet')").fetchone()[0]
    print('  ' + m + ': ' + str(n) + ' rows')

# Close connection
con.close()
print()
print('Connection closed.')

=== Clients with data in 2026-03 vs total ===


  active=55 / 104 clients (52.9%)

=== Content items in 2026-03 vs dim_content ===


  active=331437 / 519606 content items (63.8%)

=== ga4 flag distribution (2026-03) ===


  ga4_true=413966 ga4_false=6408671 ga4_null=3018741

=== Monthly row counts (unbalanced panel) ===


  2025-01: 1297 rows


  2025-06: 329201 rows


  2025-12: 7752930 rows


  2026-03: 9841378 rows


  2026-06: 11694072 rows

Connection closed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ - then submit your repo URL on the card. Done.

**Verified windows (Option 1):**
- Feature window: 2026-03-01 -> 2026-05-31 (daily facts, aggregated per content)
- Prediction cutoff: 2026-05-31
- Label window: 2026-06-01 -> 2026-06-30 (June decline)
- Query table window: 2026-04-02 -> 2026-06-30 (only *_prev30 columns safe as features)